In [ ]:
import pandas as pd
import os
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PATH_TO_DATA = '/content/drive/MyDrive'

# --- Наборы тикеров ---
# CORE: основные нефтяные «голубые фишки» (объект исследования) -> full_df.csv
CORE_TICKERS = ['LKOH_1day', 'ROSN_1day', 'TATN_1day', 'SIBN_1day', 'SNGS_1day']
#CORE_TICKERS = ['LKOH_D1', 'ROSN_D1', 'TATN_D1', 'SIBN_D1', 'SNGS_D1']
# SECTOR: расширенный нефтегазовый пул для pooled/предобучения (нужны соответствующие _1day.csv)
SECTOR_TICKERS = CORE_TICKERS + ['TATNP_1day', 'SNGSP_1day', 'BANE_1day', 'BANEP_1day', 'GAZP_1day', 'NVTK_1day']
#SECTOR_TICKERS = CORE_TICKERS + ['TATNP_D1', 'SNGSP_D1', 'BANE_D1', 'BANEP_D1', 'GAZP_D1', 'NVTK_D1']

#TICKERS = CORE_TICKERS  # для второго датасета: TICKERS = SECTOR_TICKERS
TICKERS = SECTOR_TICKERS
def create_dataset(path, tickers):
    all_frames = []

    base_cols = ['begin', 'close', 'ticker']
    #base_cols = ['datetime', 'close', 'ticker']

    # value — оборот в рублях (лучшая мера ликвидности); volume — объём в штуках.
    # Держим оба сырыми; лишний столбец можно убрать перед обучением.
    extra_cols = ['open', 'high', 'low', 'value', 'volume']

    for ticker in tickers:
        file_path = os.path.join(path, f"{ticker}.csv")

        if os.path.exists(file_path):
            df = pd.read_csv(file_path)

            df['begin'] = pd.to_datetime(df['begin'])

            #df = df[df['begin'] >= '2015-01-01']

            # оставляем только реально присутствующие столбцы (на случай отсутствия value и т.п.)
            cols_to_keep = base_cols + [col for col in extra_cols if col in df.columns]
            df = df[cols_to_keep]

            df['target'] = df['close'].shift(-1)

            df = df.dropna(subset=['target'])

            all_frames.append(df)
            print(f"Добавлен тикер {ticker}: {len(df)} строк, столбцы: {list(df.columns)}")
        else:
            print(f"ВНИМАНИЕ: файл не найден — {file_path}")

    master_df = pd.concat(all_frames, ignore_index=True)

    master_df = master_df.sort_values(by='begin')

    return master_df

df = create_dataset(PATH_TO_DATA, TICKERS)

Добавлен тикер LKOH_1day: 4087 строк, столбцы: ['begin', 'close', 'ticker', 'open', 'high', 'low', 'value', 'volume', 'target']
Добавлен тикер ROSN_1day: 4087 строк, столбцы: ['begin', 'close', 'ticker', 'open', 'high', 'low', 'value', 'volume', 'target']
Добавлен тикер TATN_1day: 4085 строк, столбцы: ['begin', 'close', 'ticker', 'open', 'high', 'low', 'value', 'volume', 'target']
Добавлен тикер SIBN_1day: 4079 строк, столбцы: ['begin', 'close', 'ticker', 'open', 'high', 'low', 'value', 'volume', 'target']
Добавлен тикер SNGS_1day: 4087 строк, столбцы: ['begin', 'close', 'ticker', 'open', 'high', 'low', 'value', 'volume', 'target']
Добавлен тикер TATNP_1day: 4085 строк, столбцы: ['begin', 'close', 'ticker', 'open', 'high', 'low', 'value', 'volume', 'target']
Добавлен тикер SNGSP_1day: 4087 строк, столбцы: ['begin', 'close', 'ticker', 'open', 'high', 'low', 'value', 'volume', 'target']
Добавлен тикер BANE_1day: 3620 строк, столбцы: ['begin', 'close', 'ticker', 'open', 'high', 'low', 'va

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44007 entries, 0 to 44006
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   begin   44007 non-null  datetime64[ns]
 1   close   44007 non-null  float64       
 2   ticker  44007 non-null  object        
 3   open    44007 non-null  float64       
 4   high    44007 non-null  float64       
 5   low     44007 non-null  float64       
 6   value   44007 non-null  float64       
 7   volume  44007 non-null  int64         
 8   target  44007 non-null  float64       
dtypes: datetime64[ns](1), float64(6), int64(1), object(1)
memory usage: 3.4+ MB


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df

,begin,close,ticker,open,high,low,value,volume,target
0,2010-02-01,1685.300,LKOH,1670.000,1689.790,1643.000,2.728247e+09,1639266,1711.000
35837,2010-02-01,189.850,GAZP,184.740,190.400,183.500,1.428144e+10,76298175,192.210
16338,2010-02-01,25.382,SNGS,25.326,25.490,24.958,6.523142e+08,25868800,25.771
20425,2010-02-01,76.480,TATNP,76.500,76.500,75.710,2.646520e+06,34781,76.780
24510,2010-02-01,14.429,SNGSP,14.276,14.441,14.270,1.276582e+08,8893600,14.530
...,...,...,...,...,...,...,...,...,...
32216,2026-01-31,1577.500,BANE,1570.000,1588.000,1570.000,1.375642e+06,872,1572.000
35836,2026-01-31,935.500,BANEP,937.000,938.500,933.500,3.092378e+06,3302,938.500
4086,2026-01-31,5306.500,LKOH,5300.000,5320.000,5293.000,2.795151e+08,52678,5316.500
24509,2026-01-31,547.900,TATNP,548.000,548.900,546.200,8.621532e+06,15741,548.900


In [ ]:
for window in [5, 10, 20]:
        # Скользящее среднее
        df[f'sma_{window}'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(window=window).mean())
        # Дистанция до средней
        df[f'dist_sma_{window}'] = (df['close'] - df[f'sma_{window}']) / df[f'sma_{window}']
df = df.dropna()
df

,begin,close,ticker,open,high,low,value,volume,target,sma_5,dist_sma_5,sma_10,dist_sma_10,sma_20,dist_sma_20
39943,2010-03-01,169.00,NVTK,169.02,171.000,167.11,1.114951e+08,660461,172.360,169.7480,-0.004407,173.9370,-0.028384,173.99350,-0.028699
19,2010-03-01,1602.37,LKOH,1590.00,1610.000,1571.00,3.542439e+09,2221187,1622.500,1574.3860,0.017775,1572.2210,0.019176,1594.99400,0.004624
24529,2010-03-01,15.12,SNGSP,14.97,15.239,14.95,1.947431e+08,12889000,15.159,14.8016,0.021511,14.5577,0.038626,14.42245,0.048366
12278,2010-03-01,140.00,SIBN,140.18,142.340,136.37,1.390710e+08,993674,140.690,138.5340,0.010582,140.2310,-0.001647,142.95650,-0.020681
8193,2010-03-01,142.08,TATN,142.34,142.340,140.51,1.924682e+08,1361057,141.780,140.1600,0.013699,140.3310,0.012463,140.16500,0.013662
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32216,2026-01-31,1577.50,BANE,1570.00,1588.000,1570.00,1.375642e+06,872,1572.000,1572.1000,0.003435,1560.1000,0.011153,1514.07500,0.041890
35836,2026-01-31,935.50,BANEP,937.00,938.500,933.50,3.092378e+06,3302,938.500,933.8000,0.001821,923.7500,0.012720,910.57500,0.027373
4086,2026-01-31,5306.50,LKOH,5300.00,5320.000,5293.00,2.795151e+08,52678,5316.500,5293.0000,0.002551,5322.0500,-0.002922,5365.60000,-0.011015
24509,2026-01-31,547.90,TATNP,548.00,548.900,546.20,8.621532e+06,15741,548.900,544.3200,0.006577,539.0600,0.016399,532.39500,0.029123


In [ ]:
!pip install yfinance moexalgo

In [ ]:
import yfinance as yf
import pandas as pd
from moexalgo import Ticker
import datetime

START_DATE = "2015-01-01"
END_DATE = datetime.datetime.now().strftime("%Y-%m-%d")

def download_extended_data():
    tickers_yf = {
        'BZ=F': 'brent',
        'RUB=X': 'usd_rub',
        'CNY=X': 'usd_cny',
        '^IRX': 'fed_rate_proxy',
        '^GSPC': 'sp500'
    }

    print("Загрузка данных из Yahoo Finance")
    yf_raw = yf.download(list(tickers_yf.keys()), start=START_DATE, end=END_DATE, auto_adjust=True)
    yf_data = yf_raw['Close'].rename(columns=tickers_yf)

    yf_data['cny_rub'] = yf_data['usd_rub'] / yf_data['usd_cny']
    yf_data = yf_data.drop(columns=['usd_cny'])
    yf_data.index = pd.to_datetime(yf_data.index).tz_localize(None)

    print("Загрузка данных с Мосбиржи (IMOEX, RGBI, RVI)")

    def get_moex_ticker(symbol, name):
        t = Ticker(symbol)
        candles = t.candles(start=START_DATE, end=END_DATE, period='1D')
        df = pd.DataFrame(candles)
        date_col = 'begin' if 'begin' in df.columns else 'date'
        df = df[[date_col, 'close']].rename(columns={date_col: 'Date', 'close': name})
        df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
        return df

    imoex_df = get_moex_ticker('IMOEX', 'imoex')
    rgbi_df = get_moex_ticker('RGBI', 'rgbi')
    rvi_df = get_moex_ticker('RVI', 'rvi')

    combined = pd.merge(imoex_df, rgbi_df, on='Date', how='outer')

    combined = pd.merge(combined, rvi_df, on='Date', how='outer')

    yf_data = yf_data.reset_index()

    '''if 'index' in yf_data.columns:
        yf_data = yf_data.rename(columns={'index': 'Date'})
    elif 'Date' not in yf_data.columns:
        yf_data.index.name = 'Date'
        yf_data = yf_data.reset_index()'''

    combined = pd.merge(combined, yf_data, on='Date', how='outer')

    combined = combined.dropna(subset=['imoex'])

    combined = combined.sort_values('Date').ffill()
    combined = combined.dropna()

    output_path = '/content/drive/MyDrive/macro_extended_v3.csv'
    combined.to_csv(output_path, index=False)

    return combined

df_macro = download_extended_data()
print(df_macro.tail())

[*********************100%***********************]  5 of 5 completed

Загрузка данных из Yahoo Finance


Загрузка данных с Мосбиржи (IMOEX, RGBI, RVI)
           Date    imoex    rgbi    rvi      brent    usd_rub        sp500  \
2997 2026-06-01  2570.04  118.64  23.55  94.980003  71.228180  7599.959961   
2998 2026-06-02  2620.39  119.04  23.97  96.000000  71.997078  7609.779785   
2999 2026-06-03  2601.35  119.05  24.50  97.809998  73.196777  7553.680176   
3000 2026-06-04  2579.90  119.11  25.58  95.029999  73.747688  7584.310059   
3001 2026-06-05  2561.04  119.14  24.56  93.089996  73.397400  7383.740234   

      fed_rate_proxy    cny_rub  
2997           3.620  10.527058  
2998           3.618  10.642584  
2999           3.623  10.824403  
3000           3.620  10.894595  
3001           3.625  10.835964  


In [ ]:
df_macro.head()

,Date,imoex,rgbi,rvi,brent,usd_rub,sp500,fed_rate_proxy,cny_rub
2,2015-01-05,1435.66,104.98,74.30,53.110001,58.750000,2020.579956,0.003,9.481770
3,2015-01-06,1480.73,104.54,72.64,51.099998,60.832001,2002.609985,0.020,9.798338
5,2015-01-08,1547.39,103.70,68.13,50.959999,62.650002,2062.139893,0.018,10.098324
6,2015-01-09,1515.37,104.47,67.23,50.110001,60.250000,2044.810059,0.015,9.712260
7,2015-01-12,1513.22,101.47,67.29,47.430000,61.430000,2028.260010,0.013,9.907585


In [ ]:
df_macro[df_macro['Date'] == '2025-04-18']

,Date,imoex,rgbi,rvi,brent,usd_rub,sp500,fed_rate_proxy,cny_rub
2694,2025-04-18,2872.77,107.99,52.08,67.959999,82.994606,5282.700195,4.205,11.350621


In [ ]:
df_macro[df_macro['Date'] == '2025-04-17']

,Date,imoex,rgbi,rvi,brent,usd_rub,sp500,fed_rate_proxy,cny_rub
2693,2025-04-17,2865.32,108.87,52.41,67.959999,82.994606,5282.700195,4.205,11.350621


In [ ]:
import requests

def get_moex_shares_dict(tickers):
    """Возвращает словарь {тикер: количество_акций}"""
    shares_map = {}

    for ticker in tickers:
        try:
            # Запрос к описанию бумаги
            url = f"https://iss.moex.com/iss/securities/{ticker}.json"
            data = requests.get(url).json()

            # Ищем параметр ISSUESIZE в секции description
            description = data['description']['data']
            for item in description:
                if item[0] == 'ISSUESIZE':
                    shares_map[ticker] = int(item[2])
                    break
        except Exception as e:
            print(f"Ошибка для {ticker}: {e}")
            shares_map[ticker] = None

    return shares_map

In [ ]:
unique_tickers = df['ticker'].unique()

shares_lookup = get_moex_shares_dict(unique_tickers)

df['shares_out'] = df['ticker'].map(shares_lookup)

df['market_cap'] = df['close'] * df['shares_out']

/tmp/ipykernel_1630/21598081.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['shares_out'] = df['ticker'].map(shares_lookup)
/tmp/ipykernel_1630/21598081.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['market_cap'] = df['close'] * df['shares_out']


In [ ]:
df[df['begin'] == '2015-03-31']

,begin,close,ticker,open,high,low,value,volume,target,sma_5,dist_sma_5,sma_10,dist_sma_10,sma_20,dist_sma_20,shares_out,market_cap
9468,2015-03-31,287.00,TATN,278.900,287.00,275.000,4.770497e+08,1702490,300.10,275.980,0.039930,280.7550,0.022244,287.37500,-0.001305,2178690700,6.252842e+11
5381,2015-03-31,252.15,ROSN,247.400,253.30,244.600,1.544288e+09,6197620,261.35,243.340,0.036204,242.0400,0.041770,248.13000,0.016201,10598177817,2.672331e+12
17632,2015-03-31,35.10,SNGS,35.690,36.96,35.000,1.241383e+09,34495800,35.99,35.395,-0.008335,35.1245,-0.000698,34.88325,0.006214,35725994705,1.253982e+12
37131,2015-03-31,138.90,GAZP,136.850,139.19,136.600,4.800951e+09,34740300,143.00,135.576,0.024518,136.2700,0.019300,142.13050,-0.022729,23673512900,3.288251e+12
21719,2015-03-31,165.70,TATNP,168.100,169.30,165.700,1.409301e+07,84180,169.60,164.100,0.009750,164.1700,0.009320,164.58500,0.006775,147508500,2.444216e+10
33052,2015-03-31,1340.00,BANEP,1331.000,1365.00,1315.000,6.436624e+07,48034,1358.00,1320.000,0.015152,1326.2000,0.010406,1291.10000,0.037875,29788012,3.991594e+10
29432,2015-03-31,1897.00,BANE,1840.000,1898.00,1832.000,9.054308e+07,48464,1855.00,1855.000,0.022642,1846.0000,0.027627,1793.10000,0.057944,147846489,2.804648e+11
1294,2015-03-31,2705.00,LKOH,2657.000,2705.00,2645.100,2.215556e+09,828403,2755.00,2651.860,0.020039,2678.3700,0.009943,2731.85000,-0.009829,692865762,1.874202e+12
41218,2015-03-31,432.00,NVTK,432.200,434.90,422.400,4.624064e+08,1079460,450.30,425.760,0.014656,435.7000,-0.008492,461.46500,-0.063851,3036306000,1.311684e+12
25804,2015-03-31,44.50,SNGSP,43.195,45.38,43.055,3.351950e+09,75163800,45.35,42.734,0.041325,42.1375,0.056066,41.42275,0.074289,7701998235,3.427389e+11


In [ ]:
df[df['begin'] == '2015-06-30']

,begin,close,ticker,open,high,low,value,volume,target,sma_5,dist_sma_5,sma_10,dist_sma_10,sma_20,dist_sma_20,shares_out,market_cap
37192,2015-06-30,145.85,GAZP,143.500,145.95,142.25,3.826293e+09,26523160,145.50,145.000,0.005862,146.1650,-0.002155,144.8245,0.007081,23673512900,3.452782e+12
41279,2015-06-30,557.70,NVTK,550.000,565.70,546.30,9.264609e+08,1661280,560.60,545.380,0.022590,544.9900,0.023322,543.9700,0.025240,3036306000,1.693348e+12
29493,2015-06-30,1907.00,BANE,1899.000,1919.00,1863.00,9.303891e+06,4915,1931.00,1903.200,0.001997,1929.1000,-0.011456,1930.1500,-0.011994,147846489,2.819433e+11
9529,2015-06-30,296.50,TATN,294.750,299.55,290.05,7.859166e+08,2666660,294.80,294.920,0.005357,294.4750,0.006877,295.2575,0.004208,2178690700,6.459818e+11
17693,2015-06-30,33.10,SNGS,33.170,33.30,32.58,7.219827e+08,21871200,32.65,33.516,-0.012412,33.6715,-0.016973,32.8090,0.008870,35725994705,1.182530e+12
1355,2015-06-30,2469.90,LKOH,2413.000,2473.00,2401.80,3.002506e+09,1226106,2429.20,2426.120,0.018045,2451.2000,0.007629,2490.0200,-0.008080,692865762,1.711309e+12
21780,2015-06-30,156.10,TATNP,157.600,158.90,156.00,4.042160e+07,257400,157.70,157.200,-0.006997,156.7200,-0.003956,155.7500,0.002247,147508500,2.302608e+10
33113,2015-06-30,1429.00,BANEP,1393.000,1430.00,1373.00,2.334743e+07,16584,1400.00,1405.200,0.016937,1428.2000,0.000560,1454.7500,-0.017701,29788012,4.256707e+10
25865,2015-06-30,42.65,SNGSP,42.605,42.80,42.28,9.427640e+08,22155800,41.99,42.490,0.003766,42.1520,0.011814,40.8590,0.043834,7701998235,3.284902e+11
13614,2015-06-30,136.00,SIBN,134.000,137.30,133.60,8.129519e+07,598320,136.40,135.480,0.003838,136.8800,-0.006429,137.2700,-0.009252,4741299639,6.448168e+11


In [ ]:
df[df['begin'] == '2015-09-30']

,begin,close,ticker,open,high,low,value,volume,target,sma_5,dist_sma_5,sma_10,dist_sma_10,sma_20,dist_sma_20,shares_out,market_cap
33179,2015-09-30,1435.00,BANEP,1448.00,1454.000,1434.00,1.391390e+07,9631,1430.000,1447.000,-0.008293,1456.7000,-0.014897,1448.50000,-0.009320,29788012,4.274580e+10
41345,2015-09-30,597.00,NVTK,588.50,604.900,586.20,8.939680e+08,1492870,587.800,590.300,0.011350,604.9100,-0.013076,612.67500,-0.025585,3036306000,1.812675e+12
9595,2015-09-30,308.55,TATN,301.80,308.550,300.60,5.080449e+08,1663210,306.650,302.770,0.019090,308.4750,0.000243,309.62000,-0.003456,2178690700,6.722350e+11
25931,2015-09-30,39.50,SNGSP,39.17,39.565,38.71,1.044354e+09,26659600,38.875,39.160,0.008682,40.1715,-0.016716,40.37900,-0.021769,7701998235,3.042289e+11
13680,2015-09-30,145.90,SIBN,145.80,147.000,145.00,2.844941e+07,194830,145.000,145.660,0.001648,146.3900,-0.003347,147.92000,-0.013656,4741299639,6.917556e+11
5508,2015-09-30,242.95,ROSN,240.00,246.200,239.50,1.407059e+09,5780490,237.900,238.990,0.016570,243.4700,-0.002136,245.43750,-0.010135,10598177817,2.574827e+12
17759,2015-09-30,33.50,SNGS,33.40,33.780,33.19,5.730330e+08,17099500,33.500,33.229,0.008156,33.6460,-0.004339,34.20675,-0.020661,35725994705,1.196821e+12
37258,2015-09-30,134.55,GAZP,133.80,135.730,132.80,5.852753e+09,43444970,132.900,132.940,0.012111,135.7150,-0.008584,139.40900,-0.034854,23673512900,3.185271e+12
29559,2015-09-30,1730.00,BANE,1739.00,1740.000,1717.00,4.078759e+07,23640,1719.000,1716.600,0.007806,1738.9000,-0.005118,1763.60000,-0.019052,147846489,2.557744e+11
1421,2015-09-30,2242.90,LKOH,2262.20,2292.000,2215.50,3.359526e+09,1486011,2184.000,2234.160,0.003912,2277.3600,-0.015132,2379.94000,-0.057581,692865762,1.554029e+12


In [ ]:
df['shares_out'].unique()

array([ 3036306000,   692865762,  7701998235,  4741299639,  2178690700,
         147508500, 35725994705, 10598177817, 23673512900,   147846489,
          29788012])

In [ ]:
import yfinance as yf

ticker = yf.Ticker("LKOH.ME")
splits = ticker.splits

if not splits.empty:
    print("Были сплиты:")
    print(splits)
else:
    print("Сплитов не зафиксировано")

Сплитов не зафиксировано


In [ ]:
import yfinance as yf

ticker = yf.Ticker("SIBN.ME")
splits = ticker.splits

if not splits.empty:
    print("Были сплиты:")
    print(splits)
else:
    print("Сплитов не зафиксировано")

Сплитов не зафиксировано


In [ ]:
import yfinance as yf

ticker = yf.Ticker("SNGS.ME")
splits = ticker.splits

if not splits.empty:
    print("Были сплиты:")
    print(splits)
else:
    print("Сплитов не зафиксировано")

Сплитов не зафиксировано


In [ ]:
import yfinance as yf

ticker = yf.Ticker("TATN.ME")
splits = ticker.splits

if not splits.empty:
    print("Были сплиты:")
    print(splits)
else:
    print("Сплитов не зафиксировано")

Сплитов не зафиксировано


In [ ]:
import yfinance as yf

ticker = yf.Ticker("ROSN.ME")
splits = ticker.splits

if not splits.empty:
    print("Были сплиты:")
    print(splits)
else:
    print("Сплитов не зафиксировано")

Сплитов не зафиксировано


In [ ]:
df_m2 = pd.read_csv('/content/drive/MyDrive/M2.csv')
df_key_rate = pd.read_csv('/content/drive/MyDrive/key_rate.csv')
df_spread = pd.read_csv('/content/drive/MyDrive/brent_urals_price_spread.csv')
df_divs = pd.read_csv('/content/drive/MyDrive/dividends(1).csv')
df_ROE = pd.read_csv('/content/drive/MyDrive/ROE.csv', encoding='windows-1251')
df_ROE = df_ROE[['ticker', 'ROE', 'publication_date']]
df_ROE.head()


,ticker,ROE,publication_date
0,ROSN,0.002,13.04.2026
1,ROSN,0.003,09.12.2025
2,ROSN,0.008,01.09.2025
3,ROSN,0.019,30.05.2025
4,ROSN,0.017,20.03.2025


In [ ]:
df_divs.head()

,secid,isin,registryclosedate,value,currencyid
0,ABIO,RU000A0JNAB6,07 19 2023,1.00,RUB
1,ABIO,RU000A0JNAB6,06 17 2024,1.20,RUB
2,ABRD,RU000A0JS5T7,07 10 2019,2.86,RUB
3,ABRD,RU000A0JS5T7,10 19 2020,1.03,RUB
4,ABRD,RU000A0JS5T7,07 12 2021,2.86,RUB


In [ ]:
target_tickers = ['ROSN', 'SNGS', 'LKOH', 'TATN', 'SIBN', 'TATNP', 'SNGSP', 'BANE', 'BANEP', 'GAZP', 'NVTK']

def to_dt(dataframe, col):
    dataframe[col] = pd.to_datetime(dataframe[col], dayfirst=True)
    return dataframe

df = df.rename(columns={'begin': 'Date'})
df = to_dt(df, 'Date')

df_macro = to_dt(df_macro, 'Date')

df_m2 = to_dt(df_m2, 'Date')

df_key_rate = df_key_rate.rename(columns={'date': 'Date'})
df_key_rate = to_dt(df_key_rate, 'Date')

df_spread = to_dt(df_spread, 'Date')

# df_divs = df_divs.rename(columns={'registryclosedate': 'Date'})
# df_divs = to_dt(df_divs, 'Date')
# df_divs = df_divs.rename(columns={'secid': 'ticker'})

df_divs = df_divs.rename(columns={'registryclosedate': 'Date'})
df_divs = to_dt(df_divs, 'Date')
df_divs = df_divs.rename(columns={'secid': 'ticker'})
df_divs = df_divs.rename(columns={'value': 'dividend_val'})   # <-- сумма дивиденда (чтобы не конфликтовать с оборотом 'value')
df_divs = df_divs[df_divs['ticker'].isin(target_tickers)]

df_ROE = df_ROE.rename(columns={'publication_date': 'Date'})
df_ROE = to_dt(df_ROE, 'Date')

/tmp/ipykernel_1630/2634167805.py:4: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dataframe[col] = pd.to_datetime(dataframe[col], dayfirst=True)
/tmp/ipykernel_1630/2634167805.py:4: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dataframe[col] = pd.to_datetime(dataframe[col], dayfirst=True)
/tmp/ipykernel_1630/2634167805.py:4: UserWarning: Parsing dates in %m %d %Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dataframe[col] = pd.to_datetime(dataframe[col], dayfirst=True)


In [ ]:
pref_map = {'TATN': 'TATNP', 'BANE': 'BANEP', 'SNGS': 'SNGSP'}  # обычка -> преф
extra = []
for base, pref in pref_map.items():
    sub = df_ROE[df_ROE['ticker'] == base].copy()
    if not sub.empty:
        sub['ticker'] = pref
        extra.append(sub)
if extra:
    df_ROE = pd.concat([df_ROE] + extra, ignore_index=True)

In [ ]:
df_spread.head()

,Date,Brent,Urals,Price_Spread,Spread_Percent
0,2022-02-21,97.26,93.44,3.82,3.93
1,2022-02-25,99.36,91.41,7.95,8.00
2,2022-03-03,109.37,92.66,16.71,15.28
3,2022-03-08,124.07,99.80,24.27,19.56
4,2022-03-11,122.29,97.72,24.57,20.09


In [ ]:
full_df = pd.merge(df, df_macro, on='Date', how='left')

full_df = pd.merge(full_df, df_m2, on='Date', how='left')

# Ключевая ставка датирована датами изменений -> привязываем ПОСЛЕДНЮЮ известную ставку
# через merge_asof(backward), без заглядывания в будущее (ставка действует до следующего изменения).
full_df = full_df.sort_values('Date')
df_key_rate = df_key_rate.sort_values('Date')
full_df = pd.merge_asof(full_df, df_key_rate, on='Date', direction='backward')

df_spread = df_spread.rename(columns={'Brent': 'Brent_raw', 'Urals': 'Urals_price'})
full_df = pd.merge(full_df, df_spread, on='Date', how='left')

# Дивиденды — событие на конкретную дату закрытия реестра (разреженно; ниже заполним 0).
full_df = pd.merge(full_df, df_divs, on=['Date', 'ticker'], how='left')

# ROE датирован датой ПУБЛИКАЦИИ отчёта -> привязываем последний известный ROE по каждому
# тикеру через merge_asof(backward, by='ticker').
full_df = full_df.sort_values('Date')
df_ROE = df_ROE.dropna(subset=['Date']).sort_values('Date')
full_df = pd.merge_asof(full_df, df_ROE, on='Date', by='ticker', direction='backward')

full_df.head()

,Date,close,ticker,open,high,low,value,volume,target,sma_5,...,rate,is_changed,Brent_raw,Urals_price,Price_Spread,Spread_Percent,isin,dividend_val,currencyid,ROE
0,2010-03-01,169.00,NVTK,169.02,171.000,167.11,1.114951e+08,660461,172.360,169.7480,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-03-01,1602.37,LKOH,1590.00,1610.000,1571.00,3.542439e+09,2221187,1622.500,1574.3860,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-03-01,15.12,SNGSP,14.97,15.239,14.95,1.947431e+08,12889000,15.159,14.8016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-03-01,140.00,SIBN,140.18,142.340,136.37,1.390710e+08,993674,140.690,138.5340,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-03-01,142.08,TATN,142.34,142.340,140.51,1.924682e+08,1361057,141.780,140.1600,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(f"Итоговая размерность: {full_df.shape}")

Итоговая размерность: (43798, 36)


In [ ]:
full_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43798 entries, 0 to 43797
Data columns (total 36 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            43798 non-null  datetime64[ns]
 1   close           43798 non-null  float64       
 2   ticker          43798 non-null  object        
 3   open            43798 non-null  float64       
 4   high            43798 non-null  float64       
 5   low             43798 non-null  float64       
 6   value           43798 non-null  float64       
 7   volume          43798 non-null  int64         
 8   target          43798 non-null  float64       
 9   sma_5           43798 non-null  float64       
 10  dist_sma_5      43798 non-null  float64       
 11  sma_10          43798 non-null  float64       
 12  dist_sma_10     43798 non-null  float64       
 13  sma_20          43798 non-null  float64       
 14  dist_sma_20     43798 non-null  float64       
 15  sh

In [ ]:
full_df['currencyid'].value_counts()

,count
currencyid,
RUB,183


In [ ]:
full_df = full_df.drop(columns=['isin', 'currencyid', 'Brent_raw', 'Urals_price', 'Price_Spread'])

In [ ]:
full_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43798 entries, 0 to 43797
Data columns (total 31 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            43798 non-null  datetime64[ns]
 1   close           43798 non-null  float64       
 2   ticker          43798 non-null  object        
 3   open            43798 non-null  float64       
 4   high            43798 non-null  float64       
 5   low             43798 non-null  float64       
 6   value           43798 non-null  float64       
 7   volume          43798 non-null  int64         
 8   target          43798 non-null  float64       
 9   sma_5           43798 non-null  float64       
 10  dist_sma_5      43798 non-null  float64       
 11  sma_10          43798 non-null  float64       
 12  dist_sma_10     43798 non-null  float64       
 13  sma_20          43798 non-null  float64       
 14  dist_sma_20     43798 non-null  float64       
 15  sh

In [ ]:
full_df['ROE'] = full_df.groupby('ticker')['ROE'].ffill()

In [ ]:
full_df['M2'] = full_df['M2'].str.replace(r'\s+', '', regex=True).astype(float)

full_df['M2'] = full_df.groupby('ticker')['M2'].ffill()

full_df['m2_is_new'] = full_df.groupby('ticker')['M2'].diff().ne(0).astype(int)

full_df = full_df.dropna(subset=['M2'])

In [ ]:
full_df = full_df.dropna(subset=['close'])
full_df = full_df.dropna(subset=['imoex'])

In [ ]:
full_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30596 entries, 12469 to 43786
Data columns (total 32 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            30596 non-null  datetime64[ns]
 1   close           30596 non-null  float64       
 2   ticker          30596 non-null  object        
 3   open            30596 non-null  float64       
 4   high            30596 non-null  float64       
 5   low             30596 non-null  float64       
 6   value           30596 non-null  float64       
 7   volume          30596 non-null  int64         
 8   target          30596 non-null  float64       
 9   sma_5           30596 non-null  float64       
 10  dist_sma_5      30596 non-null  float64       
 11  sma_10          30596 non-null  float64       
 12  dist_sma_10     30596 non-null  float64       
 13  sma_20          30596 non-null  float64       
 14  dist_sma_20     30596 non-null  float64       
 15  sha

In [ ]:
full_df = full_df.rename(columns={'is_changed': 'rate_is_changed'})

full_df['dividend_val'] = full_df['dividend_val'].fillna(0)
full_df['next_div_date'] = full_df[full_df['dividend_val'] > 0]['Date']
full_df['next_div_date'] = full_df.groupby('ticker')['next_div_date'].bfill()
full_df['days_to_div'] = (full_df['next_div_date'] - full_df['Date']).dt.days.fillna(365)

full_df.head()

,Date,close,ticker,open,high,low,value,volume,target,sma_5,...,cny_rub,M2,rate,rate_is_changed,Spread_Percent,dividend_val,ROE,m2_is_new,next_div_date,days_to_div
12469,2015-01-05,142.0,SIBN,140.200,145.900,139.30,2.289392e+06,16230,144.600,142.48,...,9.48177,30142.0,17.0,0.0,NaN,0.0,0.050,0,2015-06-22,168.0
12470,2015-01-05,29.6,SNGSP,29.225,29.865,29.08,5.909840e+08,20022900,30.195,29.91,...,9.48177,30142.0,17.0,0.0,NaN,0.0,NaN,0,2015-07-16,192.0
12471,2015-01-05,238.0,TATN,228.500,238.000,225.40,2.701104e+08,1164410,228.750,235.30,...,9.48177,30142.0,17.0,0.0,NaN,0.0,0.041,0,2015-07-15,191.0
12472,2015-01-05,2295.0,LKOH,2201.000,2317.000,2200.90,1.297335e+09,567928,2345.000,2257.72,...,9.48177,30142.0,17.0,0.0,NaN,0.0,0.020,0,2015-07-14,190.0
12473,2015-01-05,134.7,TATNP,132.000,136.800,130.70,4.189670e+06,30710,135.300,134.86,...,9.48177,30142.0,17.0,0.0,NaN,0.0,0.041,0,2018-07-06,1278.0


In [ ]:
# --- Проверка качества заполнения (после исправления мержей) ---
# ROE должен быть заполнен почти везде (раньше из-за точного мержа часто получался весь NaN).
for col in ['ROE', 'imoex', 'M2']:
    if col in full_df.columns:
        print(f"{col}: доля заполненных = {full_df[col].notna().mean():.3f}")

# имя столбца ключевой ставки зависит от вашего key_rate.csv — проверьте по списку ниже
print("Столбцы full_df:", list(full_df.columns))
# при необходимости: print('ставка заполнена:', full_df['<имя_столбца_ставки>'].notna().mean())

ROE: доля заполненных = 0.802
imoex: доля заполненных = 1.000
M2: доля заполненных = 1.000
Столбцы full_df: ['Date', 'close', 'ticker', 'open', 'high', 'low', 'value', 'volume', 'target', 'sma_5', 'dist_sma_5', 'sma_10', 'dist_sma_10', 'sma_20', 'dist_sma_20', 'shares_out', 'market_cap', 'imoex', 'rgbi', 'rvi', 'brent', 'usd_rub', 'sp500', 'fed_rate_proxy', 'cny_rub', 'M2', 'rate', 'rate_is_changed', 'Spread_Percent', 'dividend_val', 'ROE', 'm2_is_new', 'next_div_date', 'days_to_div']


In [ ]:
full_df.to_csv('/content/drive/MyDrive/full_df_sector.csv', index=False)

In [ ]:
df_first = full_df[full_df['Date'].between('2015-01-01', '2021-12-31')]
df_first = df_first[df_first['ticker'] != 'SNGS']
df_first.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17650 entries, 12469 to 31883
Data columns (total 34 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             17650 non-null  datetime64[ns]
 1   close            17650 non-null  float64       
 2   ticker           17650 non-null  object        
 3   open             17650 non-null  float64       
 4   high             17650 non-null  float64       
 5   low              17650 non-null  float64       
 6   value            17650 non-null  float64       
 7   volume           17650 non-null  int64         
 8   target           17650 non-null  float64       
 9   sma_5            17650 non-null  float64       
 10  dist_sma_5       17650 non-null  float64       
 11  sma_10           17650 non-null  float64       
 12  dist_sma_10      17650 non-null  float64       
 13  sma_20           17650 non-null  float64       
 14  dist_sma_20      17650 non-null  float6